# C5-00c · CANCER — ETL, Feature Engineering y Preprocessing
### Capítulo 5 · Machine Learning 1: Aprendizaje supervisado · Samsung Innovation Campus
**Puente Probabilidad & Estadística → Machine Learning**

**Tipo de caso:** clasificación **binaria médica de alta dimensión**: el *caso de alto riesgo*.
30 features numéricas muy **correlacionadas**, muchas **asimétricas**, y errores de **costo asimétrico**
(no detectar un tumor maligno es mucho más grave que una falsa alarma).

**Objetivos**
1. ETL con una trampa clásica: **invertir el target** para que 1 = maligno.
2. Entender la estructura mean / error / worst de las 30 features.
3. Medir asimetría y aplicar log **decidido en train**.
4. Medir **multicolinealidad (VIF)** y reducirla **aprendiendo en train**.
5. Crear features de "heterogeneidad" del tumor (worst / mean).
6. Exportar el módulo `cap5_cancer.py`.

**Cómo encaja este Colab en el capítulo**

```
CAPA BASE (este Colab)                      CAPA MODELO
ETL -> Feature Engineering -> Preprocessing -> módulo cap5_cancer.py -> C5-M01 ... C5-M12
```

Al final se exporta el módulo **`cap5_cancer.py`** con la **misma interfaz** que los otros dos datasets:

| Función | Devuelve |
|---|---|
| `cargar(tarea)` | `X` (DataFrame) e `y` (Series) listos para modelar |
| `preprocesador(tarea, ...)` | un `Pipeline` / `ColumnTransformer` **sin ajustar** |
| `dividir(X, y, tarea)` | train/test (estratificado si es clasificación) |
| `INFO` | ficha del dataset: tipo de caso, tareas, métrica sugerida |

Los Colabs de modelos solo cambian `DATASET = "cancer"` y reutilizan todo esto.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

SEED = 42
pd.set_option("display.precision", 3)
pd.set_option("display.max_rows", 60)

## 2. ETL

| Etapa | Qué hacemos | Puente P&E |
|---|---|---|
| **E**xtract | `load_breast_cancer(as_frame=True)` | muestra de n = 569 tumores |
| **T**ransform | nombres con `_`, **invertir target** | definir la clase positiva = el evento de interés |
| **L**oad | CSV limpio | muestra fija y auditable |

In [ ]:
# Extract
crudo = load_breast_cancer(as_frame=True)
print("target_names de sklearn:", dict(enumerate(crudo.target_names)))
print("Conteo del target original:", crudo.frame["target"].value_counts().sort_index().to_dict())

> **Trampa:** en sklearn **0 = maligno** y **1 = benigno**. Si no lo invertimos, la "sensibilidad"
> mediría qué tan bien detectamos tumores **benignos**. La clase positiva tiene que ser el **evento de
> interés**: P(maligno | X).

In [ ]:
# Transform
df = crudo.frame.copy()
df.columns = df.columns.str.replace(" ", "_")
df["maligno"] = 1 - df.pop("target")

print("Forma:", df.shape, "| nulos:", int(df.isna().sum().sum()), "| duplicados:", df.duplicated().sum())
df["maligno"].value_counts().rename({0: "benigno (0)", 1: "maligno (1)"}).to_frame("n")

In [ ]:
# Estructura de las 30 features: 10 medidas x 3 estadísticos
medidas = ["radius", "texture", "perimeter", "area", "smoothness", "compactness",
           "concavity", "concave_points", "symmetry", "fractal_dimension"]
estructura = pd.DataFrame({
    "media (mean_)": [f"mean_{m}" in df for m in medidas],
    "error estándar (_error)": [f"{m}_error" in df for m in medidas],
    "peor valor (worst_)": [f"worst_{m}" in df for m in medidas],
}, index=medidas)
estructura

> **Puente P&E:** cada tumor tiene muchas células. Por cada medida, el dataset trae tres
> **estadísticos de la muestra de células**: la **media**, el **error estándar** de esa media y el
> **peor valor** (promedio de los 3 más extremos). El dataset ya viene "feature-engineereado" con
> estadística descriptiva.

In [ ]:
# Load
df.to_csv("cancer_limpio.csv", index=False)

## 3. EDA orientado a decisiones

### 3.1 Prior y prevalencia

In [ ]:
prev = df["maligno"].mean()
print(f"Prevalencia en el dataset: P(maligno) = {prev:.3f}")

> **Puente P&E:** la prevalencia del dataset (~37%) **no es** la de la población general.
> El valor predictivo positivo de un modelo depende de la prevalencia (teorema de Bayes, C5-M04):
> el mismo modelo en un tamizaje poblacional tendría muchos más falsos positivos relativos.

### 3.2 ¿Qué features separan maligno de benigno?

In [ ]:
X_all = df.drop(columns=["maligno"])
filas = []
for c in X_all.columns:
    a, b = df.loc[df.maligno == 1, c], df.loc[df.maligno == 0, c]
    U, p = stats.mannwhitneyu(a, b)
    filas.append({"feature": c, "mediana_maligno": a.median(), "mediana_benigno": b.median(),
                  "AUC_univariada": U / (len(a) * len(b)), "p_valor": p})
separacion = pd.DataFrame(filas).set_index("feature").sort_values("AUC_univariada", ascending=False)
separacion.head(10)

> **Puente P&E:** usamos **Mann-Whitney** (no paramétrico) porque muchas features son asimétricas.
> U / (n₁·n₀) es exactamente la **AUC de esa feature sola**: la probabilidad de que un tumor maligno
> al azar tenga un valor mayor que uno benigno al azar.

### 3.3 Asimetría

In [ ]:
asim = X_all.skew().sort_values(ascending=False)
print(f"Features con asimetría > 1: {(asim > 1).sum()} de {len(asim)}")
pd.DataFrame({"asimetria": asim, "asimetria_log1p": np.log1p(X_all[asim.index]).skew()}).head(12)

> **Decisión:** aplicar `log1p` a las features con asimetría > 1. **Cuáles** se decide en `fit`
> (solo con train) mediante el transformador `LogSesgadas`.

### 3.4 Multicolinealidad

In [ ]:
corr = X_all.corr().abs()
pares = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
             .stack().sort_values(ascending=False).rename("|correlación|"))
pares.head(10).to_frame()

In [ ]:
# VIF de las 10 features "mean_": VIF_j = diagonal de la inversa de la matriz de correlación
cols_mean = [c for c in X_all.columns if c.startswith("mean_")]
vif = pd.Series(np.diag(np.linalg.inv(X_all[cols_mean].corr().to_numpy())), index=cols_mean, name="VIF")
vif.sort_values(ascending=False).to_frame()

> **Puente P&E:** radio, perímetro y área están ligados por la **geometría** (P ≈ 2πr, A ≈ πr²):
> son casi la misma variable aleatoria. VIF en los **miles** ⇒ los coeficientes de una regresión
> o una logística tendrían **varianza enorme** y signos sin sentido.
> Solución: eliminar iterativamente la de mayor VIF hasta que todas queden < 10, **aprendiendo en train**.

## 4. Feature Engineering

| Hallazgo | Feature | Justificación P&E |
|---|---|---|
| `worst_` y `mean_` describen el extremo y el centro de la misma distribución de células | `ratio_worst_mean_<medida>` = worst / mean | mide la **heterogeneidad** del tumor (cola vs. centro), poco correlacionada con el tamaño |

Se usa para `radius, texture, smoothness, compactness` (concavity puede valer 0 ⇒ división por cero).

In [ ]:
df_fe = df.copy()
for v in ["radius", "texture", "smoothness", "compactness"]:
    df_fe[f"ratio_worst_mean_{v}"] = df_fe[f"worst_{v}"] / df_fe[f"mean_{v}"]

nuevas = [c for c in df_fe.columns if c.startswith("ratio_")]
filas = []
for c in nuevas:
    a, b = df_fe.loc[df_fe.maligno == 1, c], df_fe.loc[df_fe.maligno == 0, c]
    U, p = stats.mannwhitneyu(a, b)
    filas.append({"feature": c, "AUC_univariada": U / (len(a) * len(b)), "p_valor": p,
                  "max_|corr|_con_originales": df_fe[X_all.columns].corrwith(df_fe[c]).abs().max()})
pd.DataFrame(filas).set_index("feature")

> **Lectura:** una feature nueva vale si aporta **separación** (AUC lejos de 0.5) **sin** ser una copia
> de otra (correlación máxima no cercana a 1).

## 5. Transformadores que aprenden solo en train

`LogSesgadas` decide **qué** columnas transformar; `SelectorVIF` decide **cuáles** eliminar.
Ambas decisiones son estimaciones ⇒ se hacen en `fit` con train.

In [ ]:
class LogSesgadas(BaseEstimator, TransformerMixin):
    """Aplica log1p a las columnas con asimetría > umbral.
    Qué columnas transformar se decide en fit (solo con train)."""

    def __init__(self, umbral=1.0):
        self.umbral = umbral

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        asimetria = X.skew()
        self.columnas_ = [c for c in X.columns
                          if asimetria[c] > self.umbral and (X[c] >= 0).all()]
        self.columnas_entrada_ = list(X.columns)
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.columnas_entrada_).copy()
        for c in self.columnas_:
            X[c] = np.log1p(X[c].clip(lower=0))
        return X

    def get_feature_names_out(self, input_features=None):
        return np.array(self.columnas_entrada_)


class SelectorVIF(BaseEstimator, TransformerMixin):
    """Elimina de a una la columna de mayor VIF hasta que todas queden
    por debajo del umbral. VIF_j = diagonal de la inversa de la matriz
    de correlación. Se ajusta solo con train."""

    def __init__(self, umbral=10.0):
        self.umbral = umbral

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        columnas = list(X.columns)
        self.eliminadas_ = []
        while len(columnas) > 1:
            corr = np.corrcoef(X[columnas].to_numpy(dtype=float), rowvar=False)
            vif = np.diag(np.linalg.pinv(corr))
            i = int(np.argmax(vif))
            if vif[i] <= self.umbral:
                break
            self.eliminadas_.append((columnas[i], float(vif[i])))
            columnas.pop(i)
        self.columnas_ = columnas
        self.columnas_entrada_ = list(X.columns)
        return self

    def transform(self, X):
        return pd.DataFrame(X, columns=self.columnas_entrada_)[self.columnas_]

    def get_feature_names_out(self, input_features=None):
        return np.array(self.columnas_)

In [ ]:
X = df_fe.drop(columns=["maligno"])
y = df_fe["maligno"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

prep = Pipeline([("log", LogSesgadas(1.0)), ("vif", SelectorVIF(10.0)), ("escalar", StandardScaler())])
prep.fit(X_train)    # <- solo train

print(f"Columnas con log: {len(prep.named_steps['log'].columnas_)}")
print(f"Columnas que sobreviven al VIF: {len(prep.named_steps['vif'].columnas_)} de {X.shape[1]}")
pd.DataFrame(prep.named_steps["vif"].eliminadas_, columns=["eliminada", "VIF_al_eliminar"])

In [ ]:
# VIF final (todas < 10)
Xt = pd.DataFrame(prep.transform(X_train), columns=prep.named_steps["vif"].columnas_)
vif_final = pd.Series(np.diag(np.linalg.inv(Xt.corr().to_numpy())), index=Xt.columns, name="VIF")
vif_final.sort_values(ascending=False).to_frame()

## 6. Verificación: ¿reducir colinealidad cuesta desempeño?

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
variantes = {
    "Dummy (prior)": Pipeline([("m", DummyClassifier(strategy="most_frequent"))]),
    "Logística: solo escalar": Pipeline([("esc", StandardScaler()), ("m", LogisticRegression(max_iter=5000))]),
    "Logística: log + VIF + escalar": Pipeline([("prep", prep), ("m", LogisticRegression(max_iter=5000))]),
}
res = {}
for nombre, pipe in variantes.items():
    s = cross_validate(pipe, X_train, y_train, cv=cv, scoring=["roc_auc", "recall"])
    res[nombre] = {"AUC": s["test_roc_auc"].mean(), "AUC_desvio": s["test_roc_auc"].std(),
                   "sensibilidad": s["test_recall"].mean()}
pd.DataFrame(res).T

> **Puente P&E:** reducir la colinealidad deja la AUC prácticamente igual (la sensibilidad apenas
> varía), pero vuelve los coeficientes
> **estables e interpretables**. La multicolinealidad es un problema de **varianza de los estimadores**,
> no de capacidad predictiva. Para árboles y ensambles se puede usar `reducir_vif=False`.

## 7. Exportar el módulo `cap5_cancer.py`

| Tarea | Target | Nota |
|---|---|---|
| `clasificacion` | `maligno` (1) | métrica clave: **sensibilidad** |
| `regresion` | `mean_area` | con `excluir_geometria=True` se quitan radio/perímetro/área (si no, R² ≈ 1 trivial) |

In [ ]:
%%writefile cap5_cancer.py
"""cap5_cancer.py - ETL + FE + Preprocessing de BREAST CANCER (Capítulo 5, SIC).
Interfaz común: cargar(tarea), preprocesador(tarea, ...), dividir(...), INFO."""
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

SEED = 42
VARIABLES_RATIO = ["radius", "texture", "smoothness", "compactness"]
GEOMETRIA = [f"{p}_{v}" for v in ["radius", "perimeter", "area"] for p in ["mean", "worst"]] + \
            [f"{v}_error" for v in ["radius", "perimeter", "area"]]

INFO = {
    "nombre": "cancer",
    "tipo_caso": "clasificación binaria médica de alta dimensión, colineal y con costo asimétrico",
    "tareas": {
        "clasificacion": "y = maligno (1) vs. benigno (0)  [invertido respecto de sklearn]",
        "regresion": "y = mean_area (excluir_geometria=True la vuelve no trivial)",
    },
    "metrica_sugerida": {"clasificacion": "sensibilidad (recall) y ROC-AUC",
                         "regresion": "RMSE"},
}

class LogSesgadas(BaseEstimator, TransformerMixin):
    """Aplica log1p a las columnas con asimetría > umbral.
    Qué columnas transformar se decide en fit (solo con train)."""

    def __init__(self, umbral=1.0):
        self.umbral = umbral

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        asimetria = X.skew()
        self.columnas_ = [c for c in X.columns
                          if asimetria[c] > self.umbral and (X[c] >= 0).all()]
        self.columnas_entrada_ = list(X.columns)
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.columnas_entrada_).copy()
        for c in self.columnas_:
            X[c] = np.log1p(X[c].clip(lower=0))
        return X

    def get_feature_names_out(self, input_features=None):
        return np.array(self.columnas_entrada_)


class SelectorVIF(BaseEstimator, TransformerMixin):
    """Elimina de a una la columna de mayor VIF hasta que todas queden
    por debajo del umbral. VIF_j = diagonal de la inversa de la matriz
    de correlación. Se ajusta solo con train."""

    def __init__(self, umbral=10.0):
        self.umbral = umbral

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        columnas = list(X.columns)
        self.eliminadas_ = []
        while len(columnas) > 1:
            corr = np.corrcoef(X[columnas].to_numpy(dtype=float), rowvar=False)
            vif = np.diag(np.linalg.pinv(corr))
            i = int(np.argmax(vif))
            if vif[i] <= self.umbral:
                break
            self.eliminadas_.append((columnas[i], float(vif[i])))
            columnas.pop(i)
        self.columnas_ = columnas
        self.columnas_entrada_ = list(X.columns)
        return self

    def transform(self, X):
        return pd.DataFrame(X, columns=self.columnas_entrada_)[self.columnas_]

    def get_feature_names_out(self, input_features=None):
        return np.array(self.columnas_)


def _etl():
    df = load_breast_cancer(as_frame=True).frame
    df.columns = df.columns.str.replace(" ", "_")
    df["maligno"] = 1 - df.pop("target")      # sklearn: 0 = maligno
    return df


def _feature_engineering(df):
    df = df.copy()
    for v in VARIABLES_RATIO:
        df[f"ratio_worst_mean_{v}"] = df[f"worst_{v}"] / df[f"mean_{v}"]
    return df


def cargar(tarea="clasificacion", excluir_geometria=False):
    df = _feature_engineering(_etl())
    if tarea == "clasificacion":
        y = df["maligno"]
        X = df.drop(columns=["maligno"])
    elif tarea == "regresion":
        y = df["mean_area"]
        X = df.drop(columns=["maligno", "mean_area"])
        if excluir_geometria:
            X = X.drop(columns=[c for c in GEOMETRIA if c in X.columns])
    else:
        raise ValueError("tarea debe ser 'clasificacion' o 'regresion'")
    return X, y


def preprocesador(tarea="clasificacion", log_sesgadas=True, reducir_vif=True,
                  umbral_vif=10.0, escalar=True):
    pasos = []
    if log_sesgadas:
        pasos.append(("log", LogSesgadas(umbral=1.0)))
    if reducir_vif:
        pasos.append(("vif", SelectorVIF(umbral=umbral_vif)))
    pasos.append(("escalar", StandardScaler() if escalar else "passthrough"))
    return Pipeline(pasos)


def dividir(X, y, tarea="clasificacion", test_size=0.2):
    estratos = y if tarea == "clasificacion" else None
    return train_test_split(X, y, test_size=test_size,
                            random_state=SEED, stratify=estratos)

In [ ]:
import importlib, cap5_cancer
importlib.reload(cap5_cancer)
from cap5_cancer import cargar, preprocesador, dividir, INFO

filas = []
for tarea, kw in [("clasificacion", {}), ("regresion", {}), ("regresion", {"excluir_geometria": True})]:
    X_, y_ = cargar(tarea, **kw)
    Xtr, Xte, ytr, yte = dividir(X_, y_, tarea)
    p = preprocesador(tarea).fit(Xtr)
    filas.append({"tarea": tarea + (" sin geometría" if kw else ""), "target": y_.name,
                  "features_entrada": X_.shape[1], "features_salida": len(p.get_feature_names_out())})
pd.DataFrame(filas).set_index("tarea")

In [ ]:
X_mod, y_mod = cargar("clasificacion")
assert X_mod.equals(X) and y_mod.equals(y), "El módulo no coincide con el notebook"
print("OK: el módulo reproduce el notebook")
INFO

In [ ]:
# (Opcional) Guardar el módulo en Google Drive para reutilizarlo en los Colabs de modelos.
USAR_DRIVE = False

if USAR_DRIVE:
    from google.colab import drive
    import shutil, os
    drive.mount("/content/drive")
    destino = "/content/drive/MyDrive/SIC_cap5"
    os.makedirs(destino, exist_ok=True)
    shutil.copy("cap5_cancer.py", destino)
    print("Módulo guardado en", destino)
    # En los Colabs de modelos:
    #   import sys; sys.path.append("/content/drive/MyDrive/SIC_cap5")
    #   from cap5_cancer import cargar, preprocesador, dividir, INFO

## 8. Ejercicio propuesto

1. Probá `umbral_vif` = 5, 10 y 30: tabla de cantidad de features vs. AUC y sensibilidad. ¿Dónde está el equilibrio?
2. Cambiá el orden: VIF **antes** del log. ¿Sobreviven las mismas columnas? ¿Por qué el VIF depende de la escala de la transformación?
3. Calculá la AUC univariada de cada feature **en train** y compará el top 5 con el de toda la base. ¿Por qué el ranking de toda la base es optimista?
4. Agregá `ratio_worst_mean_concavity` con `+ 1e-3` en el denominador. ¿Es buena idea? Discutí el sesgo que introduce.

## 9. Cierre: tabla puente

| Concepto P&E | Herramienta ML en este Colab |
|---|---|
| Clase positiva = evento de interés | invertir el target (`1 - target`) |
| Prevalencia y prior | `mean()` del target; dependencia del VPP |
| Estadísticos de una muestra de células | features mean / error / worst |
| Test no paramétrico, AUC univariada | Mann-Whitney U / (n₁·n₀) |
| Asimetría | `LogSesgadas` (decide en train) |
| Multicolinealidad, varianza de estimadores | VIF = diag(R⁻¹), `SelectorVIF` |
| Heterogeneidad (cola vs. centro) | `ratio_worst_mean_*` |
| Estimadores ajustados solo en train | todo dentro de `Pipeline` |

## 10. Autoevaluación
1. ¿Qué mediría la "sensibilidad" si no invirtiéramos el target?
2. ¿Por qué Mann-Whitney y no un test t?
3. ¿Por qué la multicolinealidad no baja la AUC pero sí arruina los coeficientes?
4. ¿Por qué `SelectorVIF` tiene que ajustarse solo con train?